**Author:** Salvador Navas  
**Date:** 2025-06-27

### ESGF — Earth System Grid Federation

**ESGF** is a globally distributed infrastructure that hosts petabytes of climate model output (CMIP5, CMIP6, CORDEX, reanalysis).

#### 🆚 ESGF vs Copernicus CDS — when to use which?

| Feature | ESGF | Copernicus CDS |
|---------|------|----------------|
| Models available | **All CMIP6 (~100 models)** | Subset (~30 models) |
| Download method | OPeNDAP (lazy/remote) or HTTP | API (staged download) |
| Regional subset | Manual (spatial slicing) | Automatic (bounding box in request) |
| Best for | Full ensemble, raw NetCDF, CORDEX | Pre-subsetted downloads, simpler API |

> **Recommendation:** Use ESGF when you need the full multi-model ensemble or variables/models not in CDS.
> For a few models over a small region, CDS is simpler.

#### 🌡️ CMIP6 scenarios (SSPs)

| Scenario | Forcing 2100 | Warming estimate (likely) | Narrative |
|----------|-------------|--------------------------|----------|
| SSP1-2.6 | 2.6 W/m² | +1.0 – +1.8 °C | Strong mitigation |
| SSP2-4.5 | 4.5 W/m² | +2.1 – +3.5 °C | Middle of the road |
| SSP3-7.0 | 7.0 W/m² | +2.8 – +4.6 °C | Delayed action |
| SSP5-8.5 | 8.5 W/m² | +3.3 – +5.7 °C | Fossil-fuelled growth |

> For infrastructure design: use **SSP2-4.5** as plausible central estimate and **SSP5-8.5** for worst-case.

#### 📦 CMIP6 variables used in pyhydra

| Variable | Long name | Units | Conversion |
|----------|-----------|-------|------------|
| `pr` | Precipitation flux | kg m⁻² s⁻¹ | × 86400 → mm/day |
| `tasmax` | Daily max near-surface temperature | K | − 273.15 → °C |
| `tasmin` | Daily min near-surface temperature | K | − 273.15 → °C |

#### 🔗 Portals
- LLNL (USA): https://esgf-node.llnl.gov/search/cmip6/
- DKRZ (Germany): https://esgf-data.dkrz.de/search/cmip6-dkrz/
- CEDA (UK): https://esgf-index1.ceda.ac.uk/search/cmip6-ceda/


In [ ]:
import os
from pathlib import Path
from itertools import product

import pandas as pd
from tqdm import tqdm

from pyhydra.utils import interactive_map
from pyhydra.data_sources.climate_change import (
    get_dataset_metadata, get_all_urls,
    get_combination_if_complete, download_file, process_file,
)

# Portable repo-root / data-dir resolution
# Works in: local clone, Docker (/workspace), Azure ML
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'notebooks').exists() or (p / 'pyhydra').exists()),
    _cwd,
)
DATA_DIR = Path(os.environ.get('HYDRA_DATA_DIR', str(REPO_ROOT / 'data')))
NB_DIR   = _cwd   # notebook directory — used to locate the CSV produced by cell 5


## 🗺️ Select area of interest

Draw a **rectangle** on the map to define the spatial subset for download.

In [ ]:
coordinates_list = interactive_map(zoom=4, center=(20, 0))

---
## ⚙️ Download configuration

The main download cell does three things:

1. **Searches ESGF** for all (model, variant, variable) combinations matching your filters
2. **Checks completeness** — only downloads models that have ALL requested variables to avoid partial datasets
3. **Spatially subsets** each file to your bounding box before saving (saves disk space vs full global files)

#### Key parameters

| Parameter | Description |
|-----------|-------------|
| `experiments` | List of scenarios — include `'historical'` + future SSPs for a complete record |
| `variables` | CMIP6 short names: `'pr'`, `'tasmax'`, `'tasmin'`, `'tas'` |
| `download_base_dir` | Output folder — one NetCDF per (model, experiment, variable) |

#### Understanding the `dataset_id` format

```
CMIP6 . day . INSTITUTION . MIROC6 . historical . r1i1p1f1 . day . pr
  │      │        │           │          │            │        │    │
project freq    institute   model    experiment   variant  table var
```
- `r1i1p1f1` = **r**ealization 1, **i**nitialization 1, **p**hysics 1, **f**orcing 1 (primary ensemble member)
- Different variants of the same model are independent simulations — useful for quantifying internal variability

> ⚠️ ESGF downloads can be slow (many files, large sizes). For daily global CMIP6:
> a single model-variable-experiment can be **50–200 GB**. Always subset spatially!


In [ ]:
try:
    # === MAIN WORKFLOW ===

    experiments = ["historical", "ssp245", "ssp585"]
    variables   = ["pr", "tasmax", "tasmin"]
    models = [
        "ACCESS-CM2", "BCC-CSM2-MR", "CanESM5", "CMCC-ESM2",
        "CNRM-ESM2-1", "EC-Earth3", "MPI-ESM1-2-HR",
        "MRI-ESM2-0", "NorESM2-MM", "UKESM1-0-LL"
    ]
    # models = []  # uncomment to auto-detect from ESGF (slow without credentials)
    variants = ["r1i1p1f1", "r1i1p1f2"]

    all_records = []

    if not models or not variants:
        print("Searching for available models and variants...")
        base_filters = {
            "project": "CMIP6",
            "table_id": "day",
            "experiment_id": experiments,
            "variable_id": variables
        }
        df_base = get_dataset_metadata(base_filters, limit=3000)
        if not models:
            models = sorted(df_base["model"].unique().tolist())
        if not variants:
            variants = sorted(df_base["variant"].unique().tolist())

    print(f"Models: {models}")
    print(f"Variants: {variants}")

    for model, variant, experiment in product(models, variants, experiments):
        print(f"Checking: {model} | {variant} | {experiment}")
        combo = get_combination_if_complete(model, experiment, variant, variables)
        if combo:
            print(f"Valid: {model} {variant} {experiment}")
            for row in combo:
                urls_info = get_all_urls(row["dataset_id"])
                for u in urls_info:
                    all_records.append({
                        "model":      row["model"],
                        "experiment": row["experiment"],
                        "variant":    row["variant"],
                        "variable":   row["variable"],
                        "dataset_id": row["dataset_id"],
                        "version":    row["version"],
                        "url":        u["url"],
                        "url_type":   u["url_type"],
                    })
        else:
            print(f"Incomplete: {model} {variant} {experiment}")

    df_out = pd.DataFrame(all_records)
    _csv_path = NB_DIR / "opendap_links_final.csv"
    df_out.to_csv(_csv_path, index=False)
    print(f"Exported to '{_csv_path}' with {len(df_out)} links.")

except Exception as e:
    print(f'[ESGF] Query failed: {e}')
    print('[ESGF] ESGF nodes may be unavailable — configure ESGF credentials and retry.')
    df_out = None


In [ ]:
# Recover df_out from disk if kernel was restarted after the query cell
try:
    _df_out = df_out
except NameError:
    _df_out = None

_csv_path = NB_DIR / 'opendap_links_final.csv'
if _df_out is None and _csv_path.exists():
    _df_out = pd.read_csv(_csv_path)
    print(f"[ESGF] Loaded {len(_df_out)} links from disk: {_csv_path.name}")

# Recover bbox — from widget or manual override below
try:
    _coords = list(coordinates_list)
except NameError:
    _coords = []

# === MANUAL BBOX OVERRIDE ===
# Set to [lat_max, lon_min, lat_min, lon_max] if the map widget bbox is missing.
# Leave empty [] to require drawing on the map above.
MANUAL_BBOX = []   # e.g. [44.0, -10.0, 35.0, 3.0]  (Iberian Peninsula)

if len(_coords) < 4 and len(MANUAL_BBOX) == 4:
    _coords = MANUAL_BBOX
    print(f"[ESGF] Using manual bbox: {MANUAL_BBOX}")

if _df_out is None or len(_coords) < 4:
    print("[ESGF] Download skipped.")
    if _df_out is None:
        print("  → Run the query cell above, or place 'opendap_links_final.csv' next to this notebook.")
    if len(_coords) < 4:
        print("  → Draw a bounding box on the map, or set MANUAL_BBOX = [lat_max, lon_min, lat_min, lon_max].")
else:
    path_output = str(DATA_DIR / 'esgf' / 'day') + '/'
    os.makedirs(path_output, exist_ok=True)

    lon_min = _coords[1]
    lon_max = _coords[3]
    lat_min = _coords[2]
    lat_max = _coords[0]

    df = _df_out.copy()
    df['version_date'] = pd.to_datetime(df['version'], format='%Y%m%d', errors='coerce')
    df_latest = df.sort_values("version_date", ascending=False)

    if "url" not in df_latest.columns or "url_type" not in df_latest.columns:
        raise ValueError("CSV must contain 'url' and 'url_type' columns.")

    df_latest = df_latest[df_latest["url_type"] == "HTTPSERVER"]

    slow_nodes = [
        "esgf-data1.llnl.gov",
        "esgf-data04.diasjp.net",
        "aims3.llnl.gov",
        "esgf.ceda.ac.uk",
        "esgf-data2.llnl.gov",
        "esg.lasg.ac.cn",
    ]
    df_urls_filtered = df_latest[~df_latest['url'].apply(
        lambda u: any(node in u for node in slow_nodes)
    )]
    urls = df_urls_filtered[["url", "url_type"]].dropna().drop_duplicates().values.tolist()
    print(f"Total links to process: {len(urls)}")
    print(f"Output directory: {path_output}")

    for url, url_type in tqdm(urls):
        process_file(url, path_output, lat_min, lat_max, lon_min, lon_max, url_type)


---
## 📊 Interpreting the results

After download, you have one NetCDF per (model, experiment, variable). Next steps:

1. **Validate historical period** against ERA5 or station data (AEMET, OGIMET) — CMIP6 biases often exceed 20%
2. **Bias correction** using Quantile Delta Mapping → see `bias_correction` notebook
3. **Compute change signal**: `ΔT = mean(SSP, 2070-2100) − mean(historical, 1981-2010)`
4. **Ensemble spread**: report 10th–90th percentile across models as uncertainty range

#### Known limitations

- Spatial resolution (~1°) is too coarse for catchment-scale studies — **downscaling required**
- CMIP6 overestimates precipitation frequency and underestimates peak intensities
- Mountain regions have larger biases due to unresolved orography
- Never use raw CMIP6 output as direct forcing for hydrological models without bias correction
